# Import BAFU into a Brightway project

This notebook does one job: it imports the BAFU (UVEK LCI Data) background database into a Brightway project, so it can be used in Activity Browser. Nothing else. All foreground processes (the product being studied) are then built by hand in Activity Browser.

It uses the same import procedure worked out in the PET bottle notebook: clean the `None` fields, run bw2io's standard strategies, translate BAFU's older category names to `biosphere3` ones, check what's still unlinked, drop it, and write the database.

**How to use it**

1. Change the two settings in the next cell: the project name and the path to the extracted ecoSpold1 folder.
2. Run all cells (Kernel → Restart & Run All). It takes a minute or two.
3. Close this notebook, open Activity Browser, pick the same project, and BAFU is there.
4. Create a new database for your own processes (for example `Synthetic Shoes`) and build in that. Don't edit BAFU itself.

Run this once per project. Every Brightway project keeps its own copy of every database, so a new project needs BAFU imported again (or copy an existing project that already has it, using Activity Browser's **Duplicate** button, which is faster).

## 1. Settings

The only cell that needs editing.

In [ ]:
import os

PROJECT_NAME = "Synthetic Shoes LCA"            # new or existing Brightway project
ECOSPOLD_PATH = "~/ecoSpold files"               # folder with the extracted BAFU ecoSpold1 XML files
DB_NAME = "BAFU"                                 # name the database will get in Brightway

ECOSPOLD_PATH = os.path.expanduser(ECOSPOLD_PATH)  # turns "~" into your home folder

## 2. Open the project and install biosphere3 and the impact methods

`set_current` creates the project if it doesn't exist yet. `bw2setup()` installs `biosphere3` (the elementary flows) and the LCIA methods, and only runs if they aren't already there.

The `assert` stops the notebook if BAFU is already in this project, so it doesn't get imported twice by accident. If you do want to re-import it, delete it first with `del databases["BAFU"]`, but be aware that any of your own processes linked to it will lose those links.

In [ ]:
import bw2data
import bw2calc
import bw2io
from bw2data import projects, databases, methods, Database

projects.set_current(PROJECT_NAME)

if "biosphere3" not in databases:
    bw2io.bw2setup()

print("Project:", projects.current)
print("Databases:", list(databases))

assert DB_NAME not in databases, (
    f"'{DB_NAME}' already exists in project '{PROJECT_NAME}'. Nothing to do. "
    f"Delete it first with del databases['{DB_NAME}'] if you really want to re-import."
)

## 3. Read the ecoSpold1 files

This reads every XML file into memory. Nothing is written to the project yet.

You'll probably see a `RuntimeWarning: divide by zero encountered in log`. It's harmless for the import. It comes from a few BAFU exchanges that have a lognormal uncertainty with a mean of zero.

In [ ]:
assert os.path.isdir(ECOSPOLD_PATH), f"Folder not found: {ECOSPOLD_PATH}. Check ECOSPOLD_PATH in the settings cell."

importer = bw2io.SingleOutputEcospold1Importer(ECOSPOLD_PATH, DB_NAME)

## 4. Replace `None` fields with empty text

Some BAFU datasets have `None` in fields that the next step uses to build each activity's unique code, which makes it crash. Empty text works fine.

In [ ]:
fields_to_clean = ["name", "categories", "unit", "reference product", "location"]

for ds in importer.data:
    for field in fields_to_clean:
        if ds.get(field) is None:
            ds[field] = ""
    for exc in ds.get("exchanges", []):
        for field in fields_to_clean:
            if exc.get(field) is None:
                exc[field] = ""

## 5. Standard cleanup and first round of linking

`apply_strategies()` runs bw2io's standard ecoSpold1 cleanup (units, codes, linking BAFU processes to each other). `match_database` links emissions and resources to `biosphere3` by name and category. `statistics()` shows how many exchanges are still unlinked.

In [ ]:
importer.apply_strategies()
importer.match_database("biosphere3", fields=["name", "categories"])
importer.statistics()

## 6. Translate BAFU's older category names and link again

BAFU uses the older ecoinvent v2 category names (`'emissions to air', 'high. pop.'`), while `biosphere3` uses the newer ones (`'air', 'urban air close to ground'`). This rewrites them, then runs the matching again. The unlinked count should drop a lot.

In [ ]:
CATEGORY_FIX = {
    ("emissions to air",): ("air",),
    ("emissions to air", "high. pop."): ("air", "urban air close to ground"),
    ("emissions to air", "low. pop."): ("air", "non-urban air or from high stacks"),
    ("emissions to air", "stratosphere + troposphere"): ("air", "lower stratosphere + upper troposphere"),
    ("emissions to soil",): ("soil",),
    ("emissions to soil", "agricultural"): ("soil", "agricultural"),
    ("emissions to soil", "industrial"): ("soil", "industrial"),
    ("emissions to water",): ("water",),
    ("emissions to water", "ground-"): ("water", "ground-"),
    ("emissions to water", "ocean"): ("water", "ocean"),
    ("emissions to water", "river"): ("water", "surface water"),
    ("resources",): ("natural resource",),
    ("resources", "land"): ("natural resource", "land"),
    ("resources", "in ground"): ("natural resource", "in ground"),
    ("resources", "in air"): ("natural resource", "in air"),
    ("resources", "in water"): ("natural resource", "in water"),
    ("resources", "biotic"): ("natural resource", "biotic"),
}

fixed_count = 0
for ds in importer.data:
    for exc in ds.get("exchanges", []):
        if exc.get("type") == "biosphere":
            cats = tuple(exc.get("categories") or ())
            if cats in CATEGORY_FIX:
                exc["categories"] = CATEGORY_FIX[cats]
                fixed_count += 1

print(f"Rewrote categories on {fixed_count} exchanges")

importer.match_database("biosphere3", fields=["name", "categories"])
importer.statistics()

## 7. Check what's still unlinked before dropping it

Anything still unlinked has to be dropped for the database to be written. Look at this list before carrying on.

- **Technosphere:** a handful of BAFU processes that point to ecoinvent activities not in this project. Expected.
- **Biosphere:** flows whose names differ from the `biosphere3` spelling (metals, particulates, NMVOC, land use and so on). Dropping them means BAFU processes lose those emissions, so impact categories like toxicity, particulate matter or land use will come out too low.
- **Greenhouse gases:** the last check below. If it prints an empty list, climate change (GWP) results aren't affected by the drop. If it lists anything, stop and fix those names before continuing.

In [ ]:
unlinked = list(importer.unlinked)
bio_unlinked = [u for u in unlinked if u.get("type") == "biosphere"]
tech_unlinked = [u for u in unlinked if u.get("type") == "technosphere"]

print(f"{len(bio_unlinked)} unique unlinked biosphere flows (first 15):")
for u in bio_unlinked[:15]:
    print("  ", u.get("name"), "|", u.get("categories"), "|", u.get("unit"))

print(f"\n{len(tech_unlinked)} unique unlinked technosphere exchanges:")
for u in tech_unlinked:
    print("  ", u.get("name"), "|", u.get("location"), "|", u.get("unit"))

GHG_WORDS = ["Carbon dioxide", "Methane", "Dinitrogen monoxide", "Sulfur hexafluoride",
             "Nitrogen fluoride", "HFC", "HCFC", "CFC", "Halon", "Perfluoro", "Tetrafluoromethane", "Hexafluoroethane"]
ghg_unlinked = sorted({(u["name"], tuple(u.get("categories") or ())) for u in bio_unlinked
                       if any(w.lower() in u["name"].lower() for w in GHG_WORDS)})

print(f"\nGreenhouse gases among the unlinked flows: {len(ghg_unlinked)}")
for name, cats in ghg_unlinked:
    print("  ", name, "|", cats)

## 8. Drop the unlinked exchanges and write BAFU to the project

This is the step that actually saves BAFU. `i_am_reckless=True` is Brightway's way of making you confirm that you're deleting data on purpose. After this, BAFU shows up in Activity Browser.

`Warning: No valid output stream.` is just the progress bar complaining about Jupyter, and can be ignored.

In [ ]:
importer.drop_unlinked(i_am_reckless=True)
importer.statistics()
importer.write_database()

print("Databases now in", projects.current, ":", list(databases))

## 9. Quick check that it works

Runs one small LCA on a BAFU activity (Swiss lorry transport, 1 tonne-kilometre) to confirm the database is linked and calculable. A result of roughly 0.1 to 0.3 kg CO2-eq per tkm means everything is fine. An error or a score of 0 means something went wrong in the import.

In [ ]:
bafu = Database(DB_NAME)
print(f"{DB_NAME}: {len(bafu)} activities")

lorry = [a for a in bafu if a["name"] == "Transport, freight, lorry, fleet average" and a["location"] == "CH"]
test_activity = lorry[0] if lorry else next(iter(bafu))

gwp = next(m for m in methods
           if m[0] == "IPCC 2013" and "GWP100" in m[-1])

lca = bw2calc.LCA({test_activity: 1}, gwp)
lca.lci()
lca.lcia()

print(f"Test activity: {test_activity}")
print(f"Method: {gwp}")
print(f"Score: {lca.score:.4f} kg CO2-eq per {test_activity['unit']}")

## Next steps in Activity Browser

1. Open Activity Browser and choose the project named in the settings cell.
2. **New database…** → give it a name for your study (for example `Synthetic Shoes`).
3. Right-click that database → **New activity**, tick **Edit Activity**, set unit and location.
4. Search BAFU in the left panel, drag activities into **Technosphere Flows**, and type the amounts. Check that each row's unit matches what you mean (kilogram, kilowatt hour, ton kilometer).
5. **LCA Setup** → **New** → drag your product into *Reference flows*, drag a method from **Impact Categories** in, then **Calculate**.

Keep your own processes in their own database, and don't re-run this notebook on the same project: re-importing BAFU can break the links from your processes to it.